# RV32I CPU — XDOTP16 效益量測（PYNQ-ZU 板上版）
同一顆 CPU（IP 1.6，bitstream 不用改）跑 3 個 kernel × 4 種實作，驗證結果並量測週期數。

| 代號 | 實作 |
|---|---|
| S0 | 純標量，資料未打包（每個 int16 佔一個字，lw + mul + add） |
| S1 | 標量 + 打包資料，用 slli / srai 拆包（RV32I + MUL） |
| S2 | 標量 + 打包資料，用 Zbb sext_h 拆包（RV32I + MUL + Zbb） |
| X  | XDOTP16 + 打包資料（一條指令 = 2 個乘加） |

Kernel：`dot`（128 元素內積）、`fir`（8 taps、16 輸出）、`matmul`（8×8 int16 矩陣乘）。

**做法**
1. cell ④ **正確性**：PS 寫入資料與程式 → run → 讀回整塊 DMEM（1024 字）與預期結果逐字比對。
2. cell ⑤ **計時**：同一支 kernel 包在「重複 R 次」的迴圈裡，用 PS 端 `time.perf_counter()` 量總時間，
   換算成每次重複的 cycle 數（= 時間 × 時脈 ÷ R），與 RTL 模擬的週期數比較。R 取使總時間約 0.5 秒，
   所以 PS 讀 PC 的延遲（微秒等級）可以忽略。
3. cell ⑥ 結果總表與報告檔。

需要：IP 1.6 的 bitstream（FEATURE bit4）；cell ① 的 IMEM_BASE / DMEM_BASE 與 Address Editor 一致。


In [ ]:
# ① 載入 Overlay 與 MMIO（重新上傳 bitstream 後請先 Kernel → Restart）
import time, json
import numpy as np
from pynq import Overlay, MMIO

BIT  = "/home/xilinx/jupyter_notebooks/design_1_wrapper.bit"   # 同資料夾需有同名 .hwh
BASE = 0x80000000
SIZE = 0x10000
IMEM_BASE, DMEM_BASE, BRAM_SIZE = 0x80010000, 0x80020000, 0x1000

ol   = Overlay(BIT)
mmio = MMIO(BASE, SIZE)
imem = MMIO(IMEM_BASE, BRAM_SIZE)
dmem = MMIO(DMEM_BASE, BRAM_SIZE)

CTRL, PC, R_FEAT = 0x00, 0x04, 0x2C
feat = mmio.read(R_FEAT)
print("FEATURE = 0x%X" % feat, "（需要 bit4 = 外部 BRAM、bit1 = MUL、bit2 = XDOTP16）")
assert feat & 0x10 and feat & 0x2 and feat & 0x4, "bitstream 不是 IP 1.6（FEATURE 應為 0x17）"

# PL 時脈：Vivado 時序是以 100 MHz 簽核。PYNQ-ZU 板上 PL0 的來源是 IOPLL（≈1500 MHz），套用 .hwh 的 ÷8 後約 187.5 MHz，
# 並不是 Block Design 裡設定的 100 MHz（診斷見 docs/XDOTP16_BENCHMARK.md）。這裡明確設回 100 MHz，讓測試落在時序簽核範圍內。
from pynq import Clocks
TARGET_CLK_MHZ = 100.0      # None = 不改，維持 PYNQ 目前的時脈
mmio.write(0x00, 0x0)       # 改時脈前先讓 CPU 留在 reset
if TARGET_CLK_MHZ is not None:
    Clocks.fclk0_mhz = TARGET_CLK_MHZ
CLK_MHZ = float(Clocks.fclk0_mhz)
print("PL 時脈（fclk0）= %.3f MHz" % CLK_MHZ)
assert TARGET_CLK_MHZ is None or abs(CLK_MHZ - TARGET_CLK_MHZ) / TARGET_CLK_MHZ < 0.02, "PL 時脈沒有設成 %s MHz（讀回 %.3f）" % (TARGET_CLK_MHZ, CLK_MHZ)


In [ ]:
# ② benchmark 資料（由 tools/gen_bench_notebook.py 產生：程式、輸入、預期結果、RTL 模擬週期數）
DATA = json.loads(r'''{"kernels":{"dot":{"macs":128,"outputs":{"RES":[0,1]},"dmem_init":{"u":{"16":1509,"17":1207,"18":4294966118,"19":4294966645,"20":1198,"21":4294965557,"22":1536,"23":4294967169,"24":57,"25":1900,"26":232,"27":1240,"28":4294965769,"29":1411,"30":4294966200,"31":1588,"32":4294965525,"33":529,"34":4294966412,"35":4294966345,"36":358,"37":1484,"38":1787,"39":4294966056,"40":996,"41":210,"42":80,"43":1779,"44":4294965497,"45":4294966277,"46":1541,"47":1267,"48":4294965813,"49":1676,"50":622,"51":4294965750,"52":4294965607,"53":411,"54":1230,"55":1800,"56":4294966206,"57":485,"58":1083,"59":1353,"60":4294966164,"61":1586,"62":242,"63":53,"64":4294965553,"65":1606,"66":1010,"67":4294965690,"68":791,"69":1709,"70":813,"71":4294967283,"72":4294965297,"73":1363,"74":4294965612,"75":278,"76":4294965715,"77":822,"78":4294965996,"79":1217,"80":4294965667,"81":562,"82":387,"83":1779,"84":4294965471,"85":326,"86":1369,"87":4294965427,"88":4294966677,"89":243,"90":1933,"91":919,"92":4294965767,"93":1695,"94":4294966501,"95":4294965694,"96":312,"97":4294965533,"98":4294965620,"99":901,"100":1167,"101":4294967178,"102":121,"103":4294966231,"104":1704,"105":1067,"106":4294967235,"107":11,"108":99,"109":1451,"110":1558,"111":1924,"112":4294967060,"113":126,"114":4294966105,"115":4294965418,"116":1067,"117":937,"118":1880,"119":34,"120":343,"121":774,"122":1383,"123":4294966114,"124":4294966105,"125":1002,"126":4294965385,"127":4294965843,"128":145,"129":4294965779,"130":4294966548,"131":4294966953,"132":1671,"133":4294966118,"134":4294965450,"135":1663,"136":4294965328,"137":4294966828,"138":4294966827,"139":881,"140":4294967124,"141":4294966131,"142":1161,"143":4294965770,"144":1132,"145":412,"146":4294965608,"147":547,"148":1709,"149":4294966245,"150":1421,"151":4294965912,"152":4294966068,"153":72,"154":505,"155":4294965623,"156":144,"157":1950,"158":806,"159":4294966766,"160":330,"161":815,"162":4294966187,"163":4294966975,"164":87,"165":4294966932,"166":4294966017,"167":1781,"168":1960,"169":444,"170":4294967161,"171":4294967121,"172":4294966065,"173":1314,"174":259,"175":4294966347,"176":670,"177":4294966852,"178":4294967219,"179":1432,"180":1286,"181":4294966607,"182":4294965614,"183":4294965384,"184":4294966241,"185":1721,"186":4294966265,"187":4294965468,"188":103,"189":1706,"190":350,"191":269,"192":4294966609,"193":975,"194":4294965844,"195":1897,"196":4294966403,"197":4294966865,"198":4294965480,"199":47,"200":1157,"201":195,"202":4294966918,"203":187,"204":1077,"205":4294966964,"206":4294966843,"207":67,"208":1350,"209":4294966254,"210":1246,"211":4294966720,"212":4294965733,"213":243,"214":4294967124,"215":4294966213,"216":4294965930,"217":4294965862,"218":4294965717,"219":4294967042,"220":4294966125,"221":4294965888,"222":4294965982,"223":1202,"224":1993,"225":509,"226":1622,"227":4294965424,"228":1350,"229":1604,"230":1635,"231":431,"232":4294965494,"233":464,"234":934,"235":4294966920,"236":4294965972,"237":4294966429,"238":525,"239":1012,"240":498,"241":4294966326,"242":534,"243":4294966860,"244":4294966441,"245":4294966863,"246":4294966694,"247":1084,"248":278,"249":1968,"250":364,"251":4294966865,"252":1069,"253":4294966121,"254":4294965526,"255":357,"256":4294966017,"257":310,"258":795,"259":527,"260":4294966592,"261":552,"262":1196,"263":4294966496,"264":4294965456,"265":790,"266":146,"267":4294966611,"268":985,"269":1193,"270":4294965507,"271":4294967163},"p":{"16":79103461,"17":4252367718,"18":4181001390,"19":4286645760,"20":124518457,"21":81264872,"22":92535305,"23":104135608,"24":34732309,"25":4232707212,"26":97255782,"27":4213704443,"28":13763556,"29":116588624,"30":4228249849,"31":83035653,"32":109902389,"33":4193649262,"34":26999143,"35":117966030,"36":31849406,"37":88671291,"38":104004500,"39":3473650,"40":105314609,"41":4189717490,"42":112001815,"43":4294116141,"44":89389105,"45":18282860,"46":53934547,"47":79821548,"48":36895139,"49":116588931,"50":21428447,"51":4172481881,"52":15990165,"53":60229517,"54":111147527,"55":4190043365,"56":4179427640,"57":59111796,"58":4287235215,"59":4225171577,"60":69928616,"61":786371,"62":95092835,"63":126092822,"64":8322836,"65":4171955033,"66":61408299,"67":2230104,"68":50725207,"69":4217505127,"70":65731417,"71":4199807113,"72":4195549329,"73":4272553236,"74":4217767559,"75":109050058,"76":4264360016,"77":57802283,"78":4218683220,"79":4194960521,"80":27001964,"81":35912040,"82":4226090669,"83":4204266893,"84":4782900,"85":4185326073,"86":127795344,"87":4260234022,"88":53412170,"89":4273994667,"90":4271112279,"91":116783873,"92":29099944,"93":4283563897,"94":86178609,"95":4232773891,"96":4265869982,"97":93913011,"98":4249814278,"99":4169726318,"100":112851937,"101":4175231993,"102":111804519,"103":17629534,"104":63962449,"105":124385876,"106":4266785923,"107":3143912,"108":12780677,"109":12320390,"110":4273210421,"111":4455995,"112":4226680134,"113":4257219806,"114":15989221,"115":4224057172,"116":4201052842,"117":4278385109,"118":4202756973,"119":78838494,"120":33359817,"121":4172285526,"122":105121094,"123":28247651,"124":30472438,"125":4270326694,"126":4238211796,"127":66322957,"128":4231397874,"129":4266394134,"130":4266654889,"131":71105958,"132":128975126,"133":4266721644,"134":4217963565,"135":23460118,"136":20380417,"137":34538267,"138":36240704,"139":4242539692,"140":51837136,"141":4250075282,"142":78185433,"143":4286314755}},"expect_words":{"0":4268899688}},"fir":{"macs":128,"outputs":{"y":[48,16]},"dmem_init":{"u":{"16":821,"17":1645,"18":1397,"19":4294966127,"20":4294966154,"21":1419,"22":4294965812,"23":4294966250,"24":930,"25":1720,"26":667,"27":870,"28":406,"29":1964,"30":4294966756,"31":634,"32":4294966914,"33":4294965517,"34":4294965910,"35":1713,"36":4294966351,"37":4294966407,"38":4294966583,"39":4294966545,"40":4294967134,"41":4294965549,"42":4294966349,"43":258,"44":218,"45":4294965415,"46":1335,"47":1596},"p":{"16":107807541,"17":4218357109,"18":93059978,"19":4226480692,"20":112722850,"21":57016987,"22":128713110,"23":41614820,"24":4178443906,"25":112327318,"26":4236770383,"27":4245814583,"28":4180541278,"29":16972877,"30":4171694298,"31":104596791,"32":43714232,"33":26608486,"34":4259579820,"35":4269933178,"36":4204198157,"37":4233037489,"38":4248304775,"39":4284415249,"40":4232968493,"41":14287106,"42":87554215,"43":1596}},"expect_words":{"48":5969159,"49":4294666975,"50":4194071,"51":7580814,"52":245449,"53":567662,"54":8491420,"55":4292762264,"56":4289449817,"57":1044664,"58":3215402,"59":3654286,"60":4291452395,"61":4292245154,"62":3086751,"63":4294160419}},"matmul":{"macs":512,"outputs":{"C":[160,64]},"dmem_init":{"u":{"16":4294966207,"80":572,"17":4294965895,"81":1871,"18":977,"82":1059,"19":1447,"83":679,"20":102,"84":946,"21":1398,"85":4294966321,"22":547,"86":531,"23":760,"87":1459,"24":4294965520,"88":4294967294,"25":1889,"89":4294966005,"26":4294965434,"90":4294966244,"27":1941,"91":4294967100,"28":348,"92":357,"29":911,"93":4294965703,"30":4294965652,"94":1432,"31":4294965312,"95":4294966876,"32":706,"96":4294965462,"33":1077,"97":4294965747,"34":472,"98":1570,"35":143,"99":4294967065,"36":1024,"100":712,"37":1117,"101":539,"38":4294966411,"102":4294965532,"39":974,"103":523,"40":1995,"104":4294965792,"41":4294966343,"105":4294966761,"42":4294965833,"106":4294965751,"43":4294965691,"107":1685,"44":4294966851,"108":1617,"45":1642,"109":4294965879,"46":452,"110":4294965954,"47":4294966913,"111":1855,"48":4294966936,"112":4294966781,"49":546,"113":1948,"50":4294966000,"114":80,"51":1855,"115":4294965361,"52":1030,"116":4294966089,"53":772,"117":4294966482,"54":4294965925,"118":4294966053,"55":4294966176,"119":880,"56":1922,"120":1580,"57":179,"121":413,"58":4294966893,"122":4294966811,"59":1007,"123":1761,"60":1464,"124":812,"61":1159,"125":4294966619,"62":4294966055,"126":4294965319,"63":1188,"127":935,"64":261,"128":1693,"65":1235,"129":1881,"66":564,"130":1927,"67":931,"131":1618,"68":1642,"132":1103,"69":4294965842,"133":699,"70":1910,"134":4294965688,"71":490,"135":1801,"72":4294965846,"136":52,"73":1058,"137":4294966563,"74":358,"138":4294965702,"75":1092,"139":4294967293,"76":598,"140":4294965973,"77":4294965379,"141":1497,"78":4294967121,"142":1201,"79":4294965724,"143":4294965460},"p":{"16":4203215807,"17":94831569,"18":91619430,"19":49807907,"48":122618428,"49":44500003,"50":4231070642,"51":95617555,"20":123861264,"21":127269050,"22":59703644,"23":4165007764,"52":4210425854,"53":4282186724,"54":4190568805,"55":4267443608,"24":70582978,"25":9372120,"26":73204736,"27":63896715,"56":4193515734,"57":4279830050,"58":35324616,"59":34339100,"28":4232513483,"29":4189846089,"30":107675203,"31":4269867460,"60":4259969568,"61":110492151,"62":4202104401,"63":121633474,"32":35847832,"33":121633520,"34":50594822,"35":4221631141,"64":127729149,"65":4168155216,"66":4241685321,"67":57735973,"36":11732866,"37":66059885,"38":75957688,"39":77921063,"68":27067948,"69":115473947,"70":4250600236,"71":61339719,"40":80937221,"41":61014580,"42":4199679594,"43":32114550,"72":123274909,"73":106039175,"74":45810767,"75":118094264,"44":69401174,"45":71565670,"46":4169335382,"47":4192010065,"76":4246929460,"77":4294834630,"78":98171605,"79":4174644401}},"expect_words":{"160":4293873012,"161":4293740253,"162":5625726,"163":2175815,"164":4288804991,"165":4293507944,"166":1323859,"167":628073,"168":4292505187,"169":4291262622,"170":4294527897,"171":5605621,"172":4294792895,"173":3356225,"174":189906,"175":4056759,"176":3846604,"177":4289960692,"178":1146623,"179":941155,"180":1307430,"181":4318798,"182":9449522,"183":4290928018,"184":4289345556,"185":1113542,"186":4290429069,"187":4287669253,"188":4293373713,"189":4292884274,"190":4290376471,"191":7431975,"192":4293529622,"193":4292907984,"194":333413,"195":5711193,"196":4291370134,"197":5528896,"198":2784710,"199":1844064,"200":3020509,"201":4291359236,"202":4294777267,"203":3927124,"204":4292222902,"205":9047831,"206":11003439,"207":4291701936,"208":8389504,"209":3061025,"210":4290518454,"211":2705248,"212":4292741028,"213":1288128,"214":3964084,"215":4290219070,"216":2319018,"217":1723281,"218":209323,"219":3903825,"220":396171,"221":554876,"222":4293729105,"223":4292557568}}},"programs":[{"kernel":"dot","var":"S0","fmt":"u","spin_pc":160,"words":[67109011,603980051,536904083,531,41603,74499,4236163,40010419,5374515,4269059,8430723,42173363,7471667,8463619,12625283,44336307,9568819,12658179,16819843,46499251,11665971,16852739,21014403,48662195,13763123,21047299,25208963,50825139,15860275,25241859,29403523,52988083,17957427,29436419,33587347,55151027,20054579,33620243,4130381027,4202531,111],"rep_words":[4023,4165523,67109011,603980051,536904083,531,41603,74499,4236163,40010419,5374515,4269059,8430723,42173363,7471667,8463619,12625283,44336307,9568819,12658179,16819843,46499251,11665971,16852739,21014403,48662195,13763123,21047299,25208963,50825139,15860275,25241859,29403523,52988083,17957427,29436419,33587347,55151027,20054579,33620243,4130381027,4202531,4294938515,4094664419,111],"rep_spin_pc":176,"sim_cycles":598,"sim_rep_period":599,"n_instr":565},{"kernel":"dot","var":"S1","fmt":"p","spin_pc":224,"words":[67109011,335544595,268468627,531,41603,74499,16946067,1090769811,1090704403,16979091,1090835603,1090737427,43221939,44303411,7471667,8520243,4236675,4269571,17143443,1090967187,1090901779,17176467,1091032979,1090934803,49710771,50792243,13763123,14811699,8431747,8464643,17340819,1091164563,1091099155,17373843,1091230355,1091132179,56199603,57281075,20054579,21103155,12626819,12659715,17538195,1091361939,1091296531,17571219,1091427731,1091329555,62688435,63769907,26346035,27394611,16810131,16843027,4063272163,4202531,111],"rep_words":[4023,4165523,67109011,335544595,268468627,531,41603,74499,16946067,1090769811,1090704403,16979091,1090835603,1090737427,43221939,44303411,7471667,8520243,4236675,4269571,17143443,1090967187,1090901779,17176467,1091032979,1090934803,49710771,50792243,13763123,14811699,8431747,8464643,17340819,1091164563,1091099155,17373843,1091230355,1091132179,56199603,57281075,20054579,21103155,12626819,12659715,17538195,1091361939,1091296531,17571219,1091427731,1091329555,62688435,63769907,26346035,27394611,16810131,16843027,4063272163,4202531,4294938515,4027555555,111],"rep_spin_pc":240,"sim_cycles":854,"sim_rep_period":855,"n_instr":821},{"kernel":"dot","var":"S2","fmt":"p","spin_pc":192,"words":[67109011,335544595,268468627,531,41603,74499,1616024467,1090704403,1616057491,1090737427,43221939,44303411,7471667,8520243,4236675,4269571,1616221843,1090901779,1616254867,1090934803,49710771,50792243,13763123,14811699,8431747,8464643,1616419219,1091099155,1616452243,1091132179,56199603,57281075,20054579,21103155,12626819,12659715,1616616595,1091296531,1616649619,1091329555,62688435,63769907,26346035,27394611,16810131,16843027,4096826595,4202531,111],"rep_words":[4023,4165523,67109011,335544595,268468627,531,41603,74499,1616024467,1090704403,1616057491,1090737427,43221939,44303411,7471667,8520243,4236675,4269571,1616221843,1090901779,1616254867,1090934803,49710771,50792243,13763123,14811699,8431747,8464643,1616419219,1091099155,1616452243,1091132179,56199603,57281075,20054579,21103155,12626819,12659715,1616616595,1091296531,1616649619,1091329555,62688435,63769907,26346035,27394611,16810131,16843027,4096826595,4202531,4294938515,4061109987,111],"rep_spin_pc":208,"sim_cycles":726,"sim_rep_period":727,"n_instr":693},{"kernel":"dot","var":"X","fmt":"p","spin_pc":96,"words":[67109011,335544595,268468627,531,41603,74499,4236163,6455947,5374515,4269059,8430723,8618891,7471667,8463619,12625283,10781835,9568819,12658179,16810131,12944779,11665971,16843027,4197489891,4202531,111],"rep_words":[4023,4165523,67109011,335544595,268468627,531,41603,74499,4236163,6455947,5374515,4269059,8430723,8618891,7471667,8463619,12625283,10781835,9568819,12658179,16810131,12944779,11665971,16843027,4197489891,4202531,4294938515,4161773283,111],"rep_spin_pc":112,"sim_cycles":342,"sim_rep_period":343,"n_instr":309},{"kernel":"fir","var":"S0","fmt":"u","spin_pc":220,"words":[67117699,71312131,75506563,79700995,83895427,88089859,92284291,96478723,100663443,201326995,67142163,42627,4237059,8431491,12625923,16820355,21014787,25209219,29403651,33598083,39226163,40307635,24841011,41389107,25889587,42470579,26938163,43552051,27986739,44633523,29035315,45714995,30083891,46796467,31132467,23175203,39259955,40340275,24055603,41421747,25104179,42503219,26152755,43584691,27201331,44666163,28249907,45747635,29298483,46829107,30347059,31564323,8421523,8487315,4097874659,111],"rep_words":[4023,4165523,67117699,71312131,75506563,79700995,83895427,88089859,92284291,96478723,100663443,201326995,67142163,42627,4237059,8431491,12625923,16820355,21014787,25209219,29403651,33598083,39226163,40307635,24841011,41389107,25889587,42470579,26938163,43552051,27986739,44633523,29035315,45714995,30083891,46796467,31132467,23175203,39259955,40340275,24055603,41421747,25104179,42503219,26152755,43584691,27201331,44666163,28249907,45747635,29298483,46829107,30347059,31564323,8421523,8487315,4097874659,4294938515,4061106403,111],"rep_spin_pc":236,"sim_cycles":380,"sim_rep_period":381,"n_instr":363},{"kernel":"fir","var":"S1","fmt":"p","spin_pc":352,"words":[67118723,71313155,17207955,1090704019,1090966291,17240979,1090769811,1090999315,75507587,79702019,17274003,1090835603,1091032339,17307027,1090901395,1091065363,83886227,134218003,201326995,33587731,42627,4237059,8431491,12625923,17209491,1091098771,1090967827,39356595,40438067,17242515,1091164563,1091000851,41519539,42601011,17275539,1091230355,1091033875,43682483,44763955,17308563,1091296147,1091066899,45845427,46926899,19433651,20482227,21530803,22579379,23627955,24676531,25725107,17932323,76931,4271363,8465795,12660227,17604243,1091493523,1091362579,39751347,40832819,17634963,1090967187,1091393299,41322163,42403635,17667987,1091032979,1091426323,43485107,44566579,17701011,1091098771,1091459347,45648051,46729523,32411315,14585523,15634099,16682675,17731251,18779827,19828403,30515747,4227219,4260115,8487315,3997211363,111],"rep_words":[4023,4165523,67118723,71313155,17207955,1090704019,1090966291,17240979,1090769811,1090999315,75507587,79702019,17274003,1090835603,1091032339,17307027,1090901395,1091065363,83886227,134218003,201326995,33587731,42627,4237059,8431491,12625923,17209491,1091098771,1090967827,39356595,40438067,17242515,1091164563,1091000851,41519539,42601011,17275539,1091230355,1091033875,43682483,44763955,17308563,1091296147,1091066899,45845427,46926899,19433651,20482227,21530803,22579379,23627955,24676531,25725107,17932323,76931,4271363,8465795,12660227,17604243,1091493523,1091362579,39751347,40832819,17634963,1090967187,1091393299,41322163,42403635,17667987,1091032979,1091426323,43485107,44566579,17701011,1091098771,1091459347,45648051,46729523,32411315,14585523,15634099,16682675,17731251,18779827,19828403,30515747,4227219,4260115,8487315,3997211363,4294938515,3893337827,111],"rep_spin_pc":368,"sim_cycles":581,"sim_rep_period":582,"n_instr":564},{"kernel":"fir","var":"S2","fmt":"p","spin_pc":304,"words":[67118723,71313155,1616286355,1090966291,1616319379,1090999315,75507587,79702019,1616352403,1091032339,1616385427,1091065363,83886227,134218003,201326995,33587731,42627,4237059,8431491,12625923,1616287891,1090967827,39356595,40438067,1616320915,1091000851,41519539,42601011,1616353939,1091033875,43682483,44763955,1616386963,1091066899,45845427,46926899,19433651,20482227,21530803,22579379,23627955,24676531,25725107,17932323,76931,4271363,8465795,12660227,1616682643,1091362579,39751347,40832819,1616713363,1091393299,41322163,42403635,1616746387,1091426323,43485107,44566579,1616779411,1091459347,45648051,46729523,32411315,14585523,15634099,16682675,17731251,18779827,19828403,30515747,4227219,4260115,8487315,4030765795,111],"rep_words":[4023,4165523,67118723,71313155,1616286355,1090966291,1616319379,1090999315,75507587,79702019,1616352403,1091032339,1616385427,1091065363,83886227,134218003,201326995,33587731,42627,4237059,8431491,12625923,1616287891,1090967827,39356595,40438067,1616320915,1091000851,41519539,42601011,1616353939,1091033875,43682483,44763955,1616386963,1091066899,45845427,46926899,19433651,20482227,21530803,22579379,23627955,24676531,25725107,17932323,76931,4271363,8465795,12660227,1616682643,1091362579,39751347,40832819,1616713363,1091393299,41322163,42403635,1616746387,1091426323,43485107,44566579,1616779411,1091459347,45648051,46729523,32411315,14585523,15634099,16682675,17731251,18779827,19828403,30515747,4227219,4260115,8487315,4030765795,4294938515,3960444643,111],"rep_spin_pc":320,"sim_cycles":513,"sim_rep_period":514,"n_instr":496},{"kernel":"fir","var":"X","fmt":"p","spin_pc":144,"words":[67117699,71312131,75506563,79700995,83886227,134218003,201326995,33587731,42627,4237059,8431491,12625923,13797003,14878475,15959947,17041419,15107763,16156339,17204915,13738019,75907,4270339,8464771,12659203,17991819,19073291,20154763,21236235,19433651,20482227,21530803,17932835,4227219,4260115,8487315,4164983523,111],"rep_words":[4023,4165523,67117699,71312131,75506563,79700995,83886227,134218003,201326995,33587731,42627,4237059,8431491,12625923,13797003,14878475,15959947,17041419,15107763,16156339,17204915,13738019,75907,4270339,8464771,12659203,17991819,19073291,20154763,21236235,19433651,20482227,21530803,17932835,4227219,4260115,8487315,4164983523,4294938515,4128216803,111],"rep_spin_pc":160,"sim_cycles":249,"sim_rep_period":250,"n_instr":232},{"kernel":"matmul","var":"S0","fmt":"u","spin_pc":264,"words":[67109011,268468499,603980307,671089299,41987,4236419,8430851,12625283,16819715,21014147,25208579,29403011,335544723,107267,4301699,8497155,12691587,16886019,21080451,25274883,29469315,42140467,43221939,44566579,45648051,46729523,47810995,48892467,49973939,7537459,16974643,18023219,19071795,20120371,21168947,22217523,6463523,33663747,37858179,42052611,46247043,50441475,54635907,58830339,63024771,42666803,43748275,44829747,45911219,46992691,48074163,49155635,50237107,24841011,25889587,26938163,27986739,29035315,30083891,31132467,23241251,67207571,8553107,4064386275,33587347,4028667619,111],"rep_words":[4023,4165523,67109011,268468499,603980307,671089299,41987,4236419,8430851,12625283,16819715,21014147,25208579,29403011,335544723,107267,4301699,8497155,12691587,16886019,21080451,25274883,29469315,42140467,43221939,44566579,45648051,46729523,47810995,48892467,49973939,7537459,16974643,18023219,19071795,20120371,21168947,22217523,6463523,33663747,37858179,42052611,46247043,50441475,54635907,58830339,63024771,42666803,43748275,44829747,45911219,46992691,48074163,49155635,50237107,24841011,25889587,26938163,27986739,29035315,30083891,31132467,23241251,67207571,8553107,4064386275,33587347,4028667619,4294938515,3994000099,111],"rep_spin_pc":280,"sim_cycles":1789,"sim_rep_period":1790,"n_instr":1724},{"kernel":"matmul","var":"S1","fmt":"p","spin_pc":360,"words":[67109011,134250771,335544851,671089299,41731,4236163,16978963,1090802707,1090737299,17011987,1090868499,1090770323,8431619,12626051,17307155,1090934291,1091065491,17340179,1091000083,1091098515,201326995,107267,4301699,8497155,12691587,16980243,1091131667,1090738579,42535219,43616691,17013267,1091197459,1090771603,44698163,45779635,17308435,1091263251,1091066771,46861107,47942579,17341459,1091329043,1091099795,49024051,50105523,20515123,21563699,22612275,23660851,24709427,25758003,26806579,19046435,16887043,21081475,25275907,29470339,17637139,1091526419,1091392275,42929971,43189043,17666963,1090769811,1091426323,44270515,45615155,17701011,1091098771,1091459347,46696627,47778099,17734035,1091164563,1091492371,48859571,49941043,7278387,8326963,17764147,18812723,19861299,20909875,21958451,31629859,33653139,8553107,3997277411,16810131,3928004323,111],"rep_words":[4023,4165523,67109011,134250771,335544851,671089299,41731,4236163,16978963,1090802707,1090737299,17011987,1090868499,1090770323,8431619,12626051,17307155,1090934291,1091065491,17340179,1091000083,1091098515,201326995,107267,4301699,8497155,12691587,16980243,1091131667,1090738579,42535219,43616691,17013267,1091197459,1090771603,44698163,45779635,17308435,1091263251,1091066771,46861107,47942579,17341459,1091329043,1091099795,49024051,50105523,20515123,21563699,22612275,23660851,24709427,25758003,26806579,19046435,16887043,21081475,25275907,29470339,17637139,1091526419,1091392275,42929971,43189043,17666963,1090769811,1091426323,44270515,45615155,17701011,1091098771,1091459347,46696627,47778099,17734035,1091164563,1091492371,48859571,49941043,7278387,8326963,17764147,18812723,19861299,20909875,21958451,31629859,33653139,8553107,3997277411,16810131,3928004323,4294938515,3893336803,111],"rep_spin_pc":376,"sim_cycles":2365,"sim_rep_period":2366,"n_instr":2300},{"kernel":"matmul","var":"S2","fmt":"p","spin_pc":312,"words":[67109011,134250771,335544851,671089299,41731,4236163,1616057363,1090737299,1616090387,1090770323,8431619,12626051,1616385555,1091065491,1616418579,1091098515,201326995,107267,4301699,8497155,12691587,1616058643,1090738579,42535219,43616691,1616091667,1090771603,44698163,45779635,1616386835,1091066771,46861107,47942579,1616419859,1091099795,49024051,50105523,20515123,21563699,22612275,23660851,24709427,25758003,26806579,19046435,16887043,21081475,25275907,29470339,1616715539,1091392275,42929971,43189043,1616745363,1091426323,44270515,45615155,1616779411,1091459347,46696627,47778099,1616812435,1091492371,48859571,49941043,7278387,8326963,17764147,18812723,19861299,20909875,21958451,31629859,33653139,8553107,4030831843,16810131,3961560803,111],"rep_words":[4023,4165523,67109011,134250771,335544851,671089299,41731,4236163,1616057363,1090737299,1616090387,1090770323,8431619,12626051,1616385555,1091065491,1616418579,1091098515,201326995,107267,4301699,8497155,12691587,1616058643,1090738579,42535219,43616691,1616091667,1090771603,44698163,45779635,1616386835,1091066771,46861107,47942579,1616419859,1091099795,49024051,50105523,20515123,21563699,22612275,23660851,24709427,25758003,26806579,19046435,16887043,21081475,25275907,29470339,1616715539,1091392275,42929971,43189043,1616745363,1091426323,44270515,45615155,1616779411,1091459347,46696627,47778099,1616812435,1091492371,48859571,49941043,7278387,8326963,17764147,18812723,19861299,20909875,21958451,31629859,33653139,8553107,4030831843,16810131,3961560803,4294938515,3960443619,111],"rep_spin_pc":328,"sim_cycles":2077,"sim_rep_period":2078,"n_instr":2012},{"kernel":"matmul","var":"X","fmt":"p","spin_pc":152,"words":[67109011,134250771,335544851,671089299,41987,4236419,8430851,12625283,201326995,107267,4301699,8497155,12691587,6554379,7635851,17106955,18188427,7537459,16974643,18023219,6463523,16886019,21080451,25274883,29469315,19138827,20220299,21301771,22383243,20515123,21563699,22612275,19046947,33653139,8553107,4165049571,16810131,4129332963,111],"rep_words":[4023,4165523,67109011,134250771,335544851,671089299,41987,4236419,8430851,12625283,201326995,107267,4301699,8497155,12691587,6554379,7635851,17106955,18188427,7537459,16974643,18023219,6463523,16886019,21080451,25274883,29469315,19138827,20220299,21301771,22383243,20515123,21563699,22612275,19046947,33653139,8553107,4165049571,16810131,4129332963,4294938515,4128215779,111],"rep_spin_pc":168,"sim_cycles":989,"sim_rep_period":990,"n_instr":924}]}''')
KERNELS  = DATA["kernels"]
PROGRAMS = DATA["programs"]
print("kernel：%s；程式 %d 支" % (", ".join(KERNELS), len(PROGRAMS)))


In [ ]:
# ③ 工具函式
NOP, WORDS, M32 = 0x00000013, 1024, 0xFFFFFFFF
R_SPIN_CONFIRM = 4          # 連續 4 次讀到 PC 在 spin 位址附近才算結束（避免 wrong-path 預取造成的誤判）

def cpu_reset(): mmio.write(CTRL, 0x0)
def cpu_run():   mmio.write(CTRL, 0x1)

def bram_write(mm, words):
    arr = np.array([w & M32 for w in words], dtype=np.uint32)
    try:    mm.array[:len(arr)] = arr
    except Exception:
        for i, w in enumerate(arr): mm.write(4 * i, int(w))

def bram_read(mm, n=WORDS):
    try:    return [int(x) for x in mm.array[:n]]
    except Exception:
        return [mm.read(4 * i) for i in range(n)]

def li_words(rd, v):
    v &= M32
    lo = ((v & 0xFFF) ^ 0x800) - 0x800
    hi = ((v - lo) & M32) >> 12
    return [(hi << 12) | (rd << 7) | 0x37, ((lo & 0xFFF) << 20) | (rd << 15) | (rd << 7) | 0x13]

def dmem_image(kernel, fmt):
    img = [0] * WORDS
    for a, v in KERNELS[kernel]["dmem_init"][fmt].items(): img[int(a)] = v
    return img

def expected_image(kernel, fmt):
    img = dmem_image(kernel, fmt)
    for a, v in KERNELS[kernel]["expect_words"].items(): img[int(a)] = v
    return img

def load(words, img):
    """CPU 放在 reset：清 DMEM + 寫入資料 + 寫入程式（其餘補 NOP）"""
    cpu_reset(); time.sleep(0.001)
    bram_write(dmem, img)
    bram_write(imem, list(words) + [NOP] * (WORDS - len(words)))

def wait_spin(spin_pc, timeout=60.0):
    """輪詢 PC，直到連續 R_SPIN_CONFIRM 次都在 spin 附近；回傳經過秒數（從呼叫起算）或 None（逾時）"""
    near = {spin_pc, spin_pc + 4, spin_pc + 8}
    t0 = time.perf_counter(); ok = 0
    while True:
        if mmio.read(PC) in near:
            ok += 1
            if ok >= R_SPIN_CONFIRM: return time.perf_counter() - t0
        else:
            ok = 0
            if time.perf_counter() - t0 > timeout: return None

RESULTS = []
def check(name, ok, detail=""):
    RESULTS.append((name, bool(ok), detail))
    print(("[PASS] " if ok else "[FAIL] ") + name + (("  —  " + detail) if detail else ""))
    return ok

def diff_words(hw, exp):
    return [(i, hw[i], exp[i]) for i in range(WORDS) if hw[i] != exp[i]]


In [ ]:
# ④ 正確性：每支程式單次執行 → 讀回整塊 DMEM 與預期逐字比對
CORRECT = {}
for p in PROGRAMS:
    k, v, f = p["kernel"], p["var"], p["fmt"]
    exp = expected_image(k, f)
    load(p["words"], dmem_image(k, f))
    cpu_run()
    t = wait_spin(p["spin_pc"])
    hw = bram_read(dmem)
    bad = diff_words(hw, exp)
    ok = (t is not None) and not bad
    CORRECT[(k, v)] = ok
    check("%-6s %-2s 結果正確" % (k, v), ok,
          "逾時" if t is None else ("DMEM 不同字數=%d，前幾個 (idx,實際,預期)=%s" % (len(bad), [(i, hex(a), hex(b)) for i, a, b in bad[:3]]) if bad else "整塊 DMEM 一致"))
    cpu_reset()


In [ ]:
# ⑤ 計時：kernel 重複 R 次，量總時間，換算每次重複的 cycle 數，與 RTL 模擬比較
TARGET_CYCLES = 50_000_000          # 目標總 cycle 數 ≈ 0.5 秒（100 MHz）
TIMING = {}
for p in PROGRAMS:
    k, v, f = p["kernel"], p["var"], p["fmt"]
    period = p["sim_rep_period"]
    R = max(2, int(round(TARGET_CYCLES / period)))
    words = li_words(31, R) + p["rep_words"][2:]          # 前 2 個字是 li x31, R（R 在這裡換成大值）
    exp = expected_image(k, f)
    load(words, dmem_image(k, f))
    t0 = time.perf_counter()
    cpu_run()
    t = wait_spin(p["rep_spin_pc"], timeout=120.0)
    t1 = time.perf_counter()
    elapsed = (t1 - t0)
    hw = bram_read(dmem)
    bad = diff_words(hw, exp)
    cpu_reset()
    cyc_per_rep = elapsed * CLK_MHZ * 1e6 / R if t is not None else float("nan")
    ratio = cyc_per_rep / period
    TIMING[(k, v)] = dict(R=R, elapsed=elapsed, cyc_per_rep=cyc_per_rep, sim_period=period, ratio=ratio, ok=(t is not None and not bad))
    check("%-6s %-2s 計時：R=%d，%.3f s → %.1f cycles/rep（模擬 %d，比值 %.4f）" % (k, v, R, elapsed, cyc_per_rep, period, ratio),
          t is not None and not bad and abs(ratio - 1) < 0.02,
          "DMEM 不同字數=%d" % len(bad) if bad else "")


In [ ]:
# ⑥ 結果總表與報告檔
print("=" * 92)
print("%-7s %-3s %9s %9s %9s %9s %9s" % ("kernel", "var", "模擬cyc", "板上cyc", "比值", "vs S0", "vs S2"))
by = {(p["kernel"], p["var"]): p for p in PROGRAMS}
rows = []
for k in KERNELS:
    for v in ("S0", "S1", "S2", "X"):
        p = by[(k, v)]
        tm = TIMING.get((k, v), {})
        s0 = by[(k, "S0")]["sim_cycles"]; s2 = by[(k, "S2")]["sim_cycles"]
        rows.append({"kernel": k, "var": v, "sim_cycles": p["sim_cycles"], "n_instr": p["n_instr"],
                     "sim_rep_period": p["sim_rep_period"], "board_cycles_per_rep": tm.get("cyc_per_rep"),
                     "board_over_sim": tm.get("ratio"), "speedup_vs_S0": s0 / p["sim_cycles"], "speedup_vs_S2": s2 / p["sim_cycles"],
                     "correct": CORRECT.get((k, v)), "R": tm.get("R"), "elapsed_s": tm.get("elapsed")})
        r = rows[-1]
        print("%-7s %-3s %9d %9s %9s %8.2fx %8.2fx" % (k, v, r["sim_cycles"],
              "%.1f" % r["board_cycles_per_rep"] if r["board_cycles_per_rep"] else "-",
              "%.4f" % r["board_over_sim"] if r["board_over_sim"] else "-", r["speedup_vs_S0"], r["speedup_vs_S2"]))
print("=" * 92)
n_pass = sum(1 for _, ok, _ in RESULTS if ok)
print("總計：%d / %d 通過  →  %s" % (n_pass, len(RESULTS), "ALL PASS ✅" if n_pass == len(RESULTS) else "有失敗項目 ❌"))
print("註：模擬cyc = 單次執行（RTL 模擬）；板上cyc = 重複執行的每次週期（含重複迴圈的 addi + bne 與 taken-branch 的 2 個 bubble），與模擬的『rep 週期』比較。")

report = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "clk_mhz": CLK_MHZ,
          "results": [{"test": n, "pass": ok, "detail": d} for n, ok, d in RESULTS], "rows": rows}
with open("/home/xilinx/jupyter_notebooks/cpu_bench_report.json", "w") as fh:
    json.dump(report, fh, indent=1)
print("報告已存到 cpu_bench_report.json")
